# Partie 5 — Structured Streaming

Projet Écosystème Big Data 2 — FastFood.ma — EMSI 5ème année, 2026-2027.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = Path("/work")
sys.path[:0] = [str(root / "src"), str(root / "third_party")]


Démarrez le producteur **avant** cette cellule, dans un autre terminal :

```bash
python producer/producer.py --mode kafka --rate 20 --bootstrap localhost:9094
```

Depuis Jupyter dans Docker, le bootstrap est déjà `kafka:9092`.

La cellule lit le topic `orders` (3 partitions), affiche le brut en `append`, écrit le JSON brut dans `lake/bronze/orders_raw`, compte ville × statut en `complete`, et montre les fenêtres de 10 minutes en `update`.

Quatre requêtes = quatre `readStream`. Spark refuse de brancher deux sinks sur la même source.


In [ ]:
from fastfood.partie5 import explain_late_event, run
from fastfood.session import build_spark

explain_late_event()
spark = build_spark("fastfood-partie5-streaming", kafka=True, delta=True)
run(spark, seconds=40)


## Modèle et modes

Une requête streaming est une table infinie : source, transformations, sink. Le moteur avance par **micro-batch** (`trigger(processingTime="5 seconds")`). Le mode continu (`trigger(continuous=...)`) est un autre chemin, latence plus basse, beaucoup moins de sinks supportés. On ne l'utilise pas : Parquet, Delta et la console de ce projet sont en micro-batch.

| Mode | Ce qu'il écrit | Ici |
| --- | --- | --- |
| append | seulement les lignes nouvelles | brut Kafka, et plus tard l'or Delta |
| update | les lignes qui changent | fenêtres encore ouvertes |
| complete | toute la table, à chaque batch | compteurs ville × statut |

Le complete sur un agrégat sans fenêtre garde tout l'historique en mémoire. C'est un outil de démo, pas un sink de production.

## Watermark de 10 minutes

Le watermark vaut `max(event_time) - 10 minutes`. Un événement dont `event_time` est plus vieux est ignoré par l'agrégation fenêtrée. Le producteur en génère environ 5 %, dont certains à 15 minutes. L'exemple chiffré est imprimé au-dessus et dans `outputs/late_example.json`.

À 15 minutes de retard, l'heure d'arrivée dépasse `event_time` de 15 minutes. Dès que le micro-batch a vu des événements « à l'heure », le watermark dépasse cet `event_time`. La ligne ne rentre dans aucune fenêtre close. C'est le prix à payer pour que l'état ne grandisse pas sans fin.
